# MiniGPT on TinyStories (Colab) - BIGGER model (~27M params)
Train a bigger GPT (8 layers, 512 dims) from scratch on the **TinyStories** dataset (short, simple children's stories written by GPT-3.5/4) so it can write coherent little stories.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

What changed from the Gutenberg version:
- Dataset: `roneneldan/TinyStories` (hundreds of thousands of stories instead of 2 books)
- **BPE subword tokenizer** (no more `<unk>` words)
- Proper GPT blocks with flash attention, **fixed weight init** (your old run started at loss 84; it should now start near 8.3)
- Mixed precision, warmup + cosine LR, best-checkpoint saving, resume support

In [ ]:
!pip -q install datasets tokenizers

import os, math, time, itertools, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from contextlib import nullcontext

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU")

## Step 0: Settings (the only cell you need to tweak)

In [ ]:
# ---- data ----
N_TRAIN_STORIES = 500_000   # more = better (2.1M exist). 500k ~ 100M tokens
N_VAL_STORIES   = 2_000
VOCAB_SIZE      = 4096      # BPE vocab size (small is fine for simple English)

# ---- model (~27M parameters) ----
BLOCK_SIZE = 256            # context length in tokens
N_LAYER    = 8
N_HEAD     = 8
N_EMBD     = 512
DROPOUT    = 0.0

# ---- training ----
BATCH_SIZE = 64
MAX_STEPS  = 10000          # ~164M tokens seen. Lower to 6000 if the ETA is too long
EVAL_EVERY = 500
LOG_EVERY  = 25            # progress line every N steps
LR         = 5e-4
MIN_LR     = 6e-5
WARMUP     = 300
WEIGHT_DECAY = 0.1
GRAD_CLIP  = 1.0

# ---- saving ----
USE_DRIVE = True            # save checkpoints to Google Drive so they survive disconnects
RESUME    = False           # set True to continue from the last saved checkpoint

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/minigpt_tinystories_v2"   # separate folder: your old model is untouched
else:
    OUT_DIR = "/content/minigpt_tinystories_v2"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to:", OUT_DIR)

## Step 1: Load TinyStories
Streams just the number of stories you ask for, so there is no 2 GB download.

In [ ]:
from datasets import load_dataset

def take(split, n):
    ds = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    return [ex["text"].strip() for ex in itertools.islice(ds, n)]

train_stories = take("train", N_TRAIN_STORIES)
val_stories   = take("validation", N_VAL_STORIES)
print(f"train stories: {len(train_stories):,} | val stories: {len(val_stories):,}")
print("\nSample story:\n", train_stories[0][:600])

## Step 2: Train a BPE tokenizer
Byte-level BPE learns subword pieces from the stories, so any word can be represented. `<|endoftext|>` marks the end of each story.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

EOT = "<|endoftext|>"
tok_path = os.path.join(OUT_DIR, "tokenizer")
os.makedirs(tok_path, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train_from_iterator(
    train_stories[:100_000], vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=[EOT]
)
tokenizer.save_model(tok_path)                      # vocab.json + merges.txt
EOT_ID = tokenizer.token_to_id(EOT)
vocab_size = tokenizer.get_vocab_size()
print("Vocab size:", vocab_size, "| EOT id:", EOT_ID)

s = "Once upon a time, there was a little girl named Lily."
ids = tokenizer.encode(s).ids
print(ids, "->", tokenizer.decode(ids))

## Step 3: Tokenize everything into one long array of token IDs

In [ ]:
def build_ids(stories, chunk=5000):
    parts = []
    for i in range(0, len(stories), chunk):
        encs = tokenizer.encode_batch(stories[i:i+chunk])
        parts.append(np.concatenate([np.array(e.ids + [EOT_ID], dtype=np.uint16) for e in encs]))
    return np.concatenate(parts)

train_ids = build_ids(train_stories)
val_ids   = build_ids(val_stories)
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")
np.save(os.path.join(OUT_DIR, "train_ids.npy"), train_ids)
np.save(os.path.join(OUT_DIR, "val_ids.npy"), val_ids)
del train_stories  # free RAM

def get_batch(split):
    d = train_ids if split == "train" else val_ids
    ix = np.random.randint(0, len(d) - BLOCK_SIZE - 1, BATCH_SIZE)
    x = torch.from_numpy(np.stack([d[i:i+BLOCK_SIZE].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([d[i+1:i+BLOCK_SIZE+1].astype(np.int64) for i in ix]))
    return x.to(device), y.to(device)

## Step 4: The model
A standard GPT: token + position embeddings → transformer blocks (causal self-attention + MLP, pre-LayerNorm) → output layer tied to the token embeddings.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_head, dropout):
        super().__init__()
        self.n_head, self.dropout = n_head, dropout
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.ln2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4 * d, bias=False)
        self.fc_proj = nn.Linear(4 * d, d, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        x = x + self.drop(self.proj(y))
        x = x + self.drop(self.fc_proj(F.gelu(self.fc(self.ln2(x)))))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.wte.weight            # weight tying
        self.apply(self._init)
        for name, p in self.named_parameters():       # scale residual projections
            if name.endswith("proj.weight") or name.endswith("fc_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)   # this fixes the loss-of-84 problem

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.wte(idx) + self.wpe(torch.arange(T, device=idx.device)))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = MiniGPT(vocab_size, BLOCK_SIZE, N_LAYER, N_HEAD, N_EMBD, DROPOUT).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Expected starting loss ~ ln({vocab_size}) = {math.log(vocab_size):.2f}")

## Step 5: Train
Uses fp16/bf16 mixed precision, AdamW, warmup + cosine decay, and saves the best checkpoint to Drive.

In [ ]:
decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95))

use_cuda = device == "cuda"
amp_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(use_cuda and amp_dtype == torch.float16))
amp_ctx = torch.autocast(device_type="cuda", dtype=amp_dtype) if use_cuda else nullcontext()

def get_lr(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (1 + math.cos(math.pi * progress)) * (LR - MIN_LR)

@torch.no_grad()
def estimate_loss(n_batches=30):
    model.eval()
    out = {}
    for split in ("train", "val"):
        ls = []
        for _ in range(n_batches):
            x, y = get_batch(split)
            with amp_ctx:
                _, loss = model(x, y)
            ls.append(loss.item())
        out[split] = sum(ls) / len(ls)
    model.train()
    return out

ckpt_path = os.path.join(OUT_DIR, "minigpt_best.pt")
history = {"step": [], "train": [], "val": []}
start_step, best_val = 0, float("inf")

if RESUME and os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    start_step, best_val, history = ck["step"] + 1, ck["best_val"], ck["history"]
    print(f"Resumed from step {ck['step']}")

print("Training on:", device, "|", torch.cuda.get_device_name(0) if use_cuda else "CPU (this will be VERY slow)")
print("Tokens per step:", BATCH_SIZE * BLOCK_SIZE)
model.train()
t0 = time.time(); t_log = time.time()
for step in range(start_step, MAX_STEPS + 1):
    for g in optimizer.param_groups:
        g["lr"] = get_lr(step)

    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        l = estimate_loss()
        history["step"].append(step); history["train"].append(l["train"]); history["val"].append(l["val"])
        print(f"step {step:>5} | train {l['train']:.3f} | val {l['val']:.3f} | {time.time()-t0:5.0f}s")
        if l["val"] < best_val:
            best_val = l["val"]
            torch.save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "step": step, "best_val": best_val, "history": history,
                "config": dict(vocab_size=vocab_size, block_size=BLOCK_SIZE, n_layer=N_LAYER,
                               n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT),
            }, ckpt_path)

    if step == MAX_STEPS:
        break
    x, y = get_batch("train")
    with amp_ctx:
        _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer); scaler.update()

    if step % LOG_EVERY == 0 and step > start_step:
        if use_cuda: torch.cuda.synchronize()
        dt = (time.time() - t_log) / LOG_EVERY
        eta = dt * (MAX_STEPS - step) / 60
        print(f"  step {step:>5} | loss {loss.item():.3f} | {dt:.2f}s/step | {BATCH_SIZE*BLOCK_SIZE/dt:,.0f} tok/s | ETA {eta:.1f} min")
        t_log = time.time()

plt.plot(history["step"], history["train"], label="train")
plt.plot(history["step"], history["val"], label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("MiniGPT on TinyStories")
plt.show()
print(f"Best val loss: {best_val:.3f} (saved to {ckpt_path})")

## Step 6: Generate stories
Loads the best checkpoint. Lower `temperature` = safer and more repetitive, higher = more creative.

In [ ]:
ck = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ck["model"])
model.eval()

@torch.no_grad()
def generate(prompt, max_new_tokens=250, temperature=0.8, top_k=40):
    ids = tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        with amp_ctx:
            logits, _ = model(x[:, -BLOCK_SIZE:])
        logits = logits[:, -1, :].float() / temperature
        if top_k:
            kth = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, [-1]]
            logits[logits < kth] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        if nxt.item() == EOT_ID:
            break
        x = torch.cat([x, nxt], dim=1)
    return tokenizer.decode(x[0].tolist())

for p in ["Once upon a time, there was a little girl named",
          "One day, a small dog found a",
          "Tom and his mom went to the park. Suddenly,"]:
    print("PROMPT:", p)
    print(generate(p), "\n" + "-" * 60)

## Tips to get better stories
- **More data:** raise `N_TRAIN_STORIES` (up to ~2.1M). This helps the most.
- **More training:** raise `MAX_STEPS` to 10,000–20,000 (the best checkpoint is saved, so you can stop any time).
- **Out of memory?** Lower `BATCH_SIZE` to 32.
- **Too slow?** Lower `MAX_STEPS` (e.g. 6000) or `N_TRAIN_STORIES`.
- **Colab disconnected?** Re-run Steps 0–4, set `RESUME = True`, and run Step 5 again.
- Use a `temperature` of 0.6–0.9 for generation.